# Fidelidade $F_1$ da porta CZ em função da largura do pacote $\sigma$

Este notebook explica, expressão por expressão, tudo o que entra no gráfico gerado por `plot.py`
(dados calculados por `calcular.py`): a **fidelidade média de porta** $F_1(\phi=\pi)$ de uma porta
controlled-Z entre dois fótons **contrapropagantes** que interagem com **dois sítios cross-Kerr**
(reprodução da Fig. 5 de Brod & Combes).

**Roteiro**

| Seção | O que é explicado | Onde aparece no código |
|---|---|---|
| 1 | Parâmetros físicos | `chi1, gamma1, Delta1, omega0, L, ...` |
| 2 | Blocos básicos: $\Gamma_j$, $t_j$, $\xi$, $c$ | `Gamma1`, `phase1`, `xi_in`, `correction` |
| 3 | Matriz S de dois fótons: $K_j$ e os termos não lineares | `K1`, `term2_bracket`, `term3_bracket` |
| 4 | Amplitude $F = 1 + G$ e sua integral | `compute_G` |
| 5 | Fidelidade $F_1(\phi)$ | `F1_fidelity` |
| 6 | Grade numérica de integração | `grade` |
| 7 | Verificação: recalcular e comparar com os `.npz` | — |
| 8 | O gráfico: eixos, rótulos, título, estilo | `plot.py` |
| 9 | Limite analítico $\sigma\to 0$ (checagem do gráfico) | — |

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from scipy.integrate import simpson
from scipy.signal import fftconvolve

---
## 1. Parâmetros físicos

Cada sítio $j\in\{1,2\}$ é um ressonador/emissor com

* **largura** (taxa de decaimento) $\gamma_j$,
* **dessintonia** $\Delta_j$,
* **acoplamento não linear** (cross-Kerr) $\chi_j$.

Os dois sítios estão separados por uma distância $L$ (em unidades em que a velocidade de propagação
vale 1, de modo que o fóton de frequência $\omega$ acumula a fase $e^{i\omega L}$ ao ir de um sítio ao outro).

O pacote de onda de entrada é gaussiano, centrado em

$$
\omega_c = \Delta_1 + \omega_0 ,
$$

em que $\omega_0$ é a frequência central **relativa a $\Delta_1$**, e tem largura espectral $\sigma$
(a variável do eixo $x$ do gráfico).

O gráfico final usa $\chi_1=\chi_2=1000$ (praticamente o limite saturado, ver seção 3),
$\gamma_1=\gamma_2=1$, $\Delta_1=\Delta_2=0$, $\omega_0=0$ e varia $L$ entre as curvas.

In [ ]:
class Params:
    # Parâmetros físicos (mesmos valores de calcular.py)
    chi1 = 1000.0      # acoplamento não linear do sítio 1
    chi2 = 1000.0      # acoplamento não linear do sítio 2
    gamma1 = 1.0       # largura do sítio 1
    gamma2 = 1.0       # largura do sítio 2
    Delta1 = 0.0       # dessintonia do sítio 1
    Delta2 = 0.0       # dessintonia do sítio 2
    omega0 = 0.0       # frequência central relativa a Delta1
    L = 0.0            # separação entre os sítios
    phi = np.pi        # fase alvo da porta: phi = pi  ->  CZ

    # Parâmetros numéricos da integração (seção 6)
    k_sigma = 9.0        # meia-janela de integração, em unidades de sigma
    pts_per_width = 14   # pontos por menor escala relevante
    n_min = 2001         # número mínimo de pontos da grade
    n_max = 600001       # teto de pontos

    @property
    def omega_c(self):
        return self.Delta1 + self.omega0      # frequência central efetiva


P = Params()

---
## 2. Blocos básicos

### 2.1 Denominador de Breit–Wigner $\Gamma_j(\omega)$

$$
\boxed{\;\Gamma_j(\omega) = \frac{\gamma_j}{2} + i\,(\Delta_j - \omega)\;}
$$

É o denominador da resposta linear do sítio $j$: $\Gamma_j(\omega)=0$ em $\omega=\Delta_j - i\gamma_j/2$,
o polo lorentziano de largura $\gamma_j$. No código: `Gamma1(w)` e `Gamma2(w)`.

### 2.2 Fase de fóton único $t_j(\omega)$

Um único fóton que atravessa o sítio $j$ só ganha uma fase (não há perda para os canais externos):

$$
\boxed{\;t_j(\omega)=\frac{\Gamma_j^{*}(\omega)}{\Gamma_j(\omega)}
=\frac{\gamma_j/2 - i(\Delta_j-\omega)}{\gamma_j/2 + i(\Delta_j-\omega)}\;},
\qquad |t_j(\omega)|=1 .
$$

Para dois sítios em sequência, com a propagação $e^{i\omega L}$ entre eles, a amplitude linear de
fóton único é

$$
s(\omega)=t_1(\omega)\,t_2(\omega)\,e^{i\omega L}.
$$

No código, `phase1(w)` $=t_1 t_2$ (o fator $e^{i\omega L}$ é tratado à parte).

### 2.3 Pacote de onda gaussiano $\xi(\omega)$

$$
\boxed{\;\xi(\omega)=\left(\frac{1}{2\pi\sigma^{2}}\right)^{1/4}
\exp\!\left[-\frac{(\omega-\omega_c)^{2}}{4\sigma^{2}}\right]\;}
$$

É uma gaussiana em amplitude, com largura de **intensidade** $|\xi|^2$ igual a $\sigma$ (desvio padrão).
O prefator garante a normalização

$$
\int_{-\infty}^{\infty}|\xi(\omega)|^{2}\,d\omega
=\frac{1}{\sqrt{2\pi}\,\sigma}\int e^{-(\omega-\omega_c)^2/(2\sigma^2)}d\omega = 1 .
$$

Ambos os fótons (a e b) entram com o mesmo perfil $\xi$, então o estado de entrada é
$|\xi\xi\rangle$, com amplitude $\xi(\nu_a)\,\xi(\nu_b)$.

### 2.4 Fator de correção $c(\omega)$

A fidelidade é medida **depois de remover** a evolução linear de fóton único (que é uma fase local,
corrigível por unitárias de um qubit). Isso equivale a aplicar $S_1^\dagger(\omega_a)S_1^\dagger(\omega_b)$,
em que $S_1(\omega)=s(\omega)$ é a amplitude de fóton único da seção 2.2:

$$
\boxed{\;c(\omega)=s^{*}(\omega)=\frac{1}{s(\omega)}
=\left[t_1(\omega)\,t_2(\omega)\right]^{*}e^{-i\omega L}\;}
$$

No código, `correction(w)`. O fator $e^{-i\omega L}$ é essencial: sem ele a referência linear e os
termos não lineares ficam em convenções de fase diferentes e $S$ deixa de ser unitária para $L\neq 0$.

In [ ]:
def Gamma1(w, p=P): return p.gamma1 / 2 + 1j * (p.Delta1 - w)
def Gamma2(w, p=P): return p.gamma2 / 2 + 1j * (p.Delta2 - w)


def xi_in(w, sigma, p=P):
    '''Pacote gaussiano normalizado.'''
    return (1.0 / (2 * np.pi * sigma ** 2)) ** 0.25 * np.exp(-(w - p.omega_c) ** 2 / (4 * sigma ** 2))


def phase1(w, p=P):
    '''t1(w) * t2(w) = Gamma1* Gamma2* / (Gamma1 Gamma2).  |phase1| = 1.'''
    return (np.conj(Gamma1(w, p)) * np.conj(Gamma2(w, p))) / (Gamma1(w, p) * Gamma2(w, p))


def correction(w, p=P):
    '''c(w) = [t1 t2]* e^{-i w L}: remove a amplitude linear de fóton único.'''
    return np.conj(phase1(w, p)) * np.exp(-1j * w * p.L)


# --- checagens rápidas das propriedades enunciadas acima ---
w = np.linspace(-20, 20, 4001)
sigma_teste = 0.7
print("|t1 t2| = 1            :", np.allclose(np.abs(phase1(w)), 1.0))
print("∫|ξ|² dω = 1           :", simpson(np.abs(xi_in(w, sigma_teste)) ** 2, x=w))

---
## 3. Matriz S de dois fótons

No setor de dois fótons contrapropagantes (um fóton em cada sentido, $a$ e $b$), a matriz S
tem duas partes:

$$
S(\omega_a,\omega_b;\nu_a,\nu_b)=
\underbrace{s(\omega_a)\,s(\omega_b)\,\delta(\omega_a-\nu_a)\,\delta(\omega_b-\nu_b)}_{\text{parte linear (fótons independentes)}}
\;+\;
\underbrace{T(\omega_a,\omega_b;\nu_a,\nu_b)\;\delta(\omega_a+\omega_b-\nu_a-\nu_b)}_{\text{parte não linear (interação)}}
$$

* Os fótons entram com frequências $(\nu_a,\nu_b)$ e saem com $(\omega_a,\omega_b)$.
* A delta $\delta(\omega_a+\omega_b-\nu_a-\nu_b)$ expressa a **conservação de energia**: a interação
  redistribui energia entre os fótons, mas a soma é conservada.
* A parte não linear é a soma de duas contribuições, uma para cada sítio onde a não linearidade atua:

$$
T = T_1 + T_2 .
$$

### 3.1 Fator de saturação $K_j$

$$
\boxed{\;K_j(\omega_a,\omega_b)=\frac{1}{1+\dfrac{2i\chi_j}{\Gamma_j(\omega_a)+\Gamma_j(\omega_b)}}\;}
$$

Descreve a **resposta não linear do sítio $j$ a dois fótons**. Com a energia total
$E=\omega_a+\omega_b$, a soma vale $\Gamma_j(\omega_a)+\Gamma_j(\omega_b)=\gamma_j+i(2\Delta_j-E)$, então

$$
K_j = K_j(E)=\frac{1}{1+\dfrac{2i\chi_j}{\gamma_j+i(2\Delta_j-E)}} ,
$$

que depende **só de $E$**, não de como a energia está repartida entre os fótons. Isso é o que permite
fatorizar a integral na seção 4.

* $\chi_j\to 0$: $K_j\to 1$, mas $T_j\propto\chi_j\to0$ (sem interação, sem fase condicional);
* $\chi_j\to\infty$: $\chi_j K_j\to (\Gamma_j(\omega_a)+\Gamma_j(\omega_b))/(2i)$ — o produto $\chi_j K_j$
  **satura** e deixa de depender de $\chi_j$. Por isso $\chi=1000\gg\gamma=1$ é praticamente o
  "limite de bloqueio" (dois fótons não ocupam o sítio ao mesmo tempo).

### 3.2 Termo não linear no sítio 1

$$
\boxed{\;
T_1=-\,e^{i(\omega_a+\nu_b)L}\;\frac{i\chi_1\gamma_1^{2}}{\pi}\;
\frac{\Gamma_2^{*}(\omega_a)\,\Gamma_2^{*}(\nu_b)}{\Gamma_2(\omega_a)\,\Gamma_2(\nu_b)}\;
\frac{K_1(\omega_a,\omega_b)}{\Gamma_1(\omega_a)\Gamma_1(\omega_b)\Gamma_1(\nu_a)\Gamma_1(\nu_b)}\;}
$$

Leitura de cada fator:

| Fator | Significado |
|---|---|
| $-\,i\chi_1\gamma_1^{2}/\pi$ | prefator da interação: força $\chi_1$, acoplamento $\gamma_1$ de cada um dos 4 "vértices" (2 fótons entrando, 2 saindo) e $1/\pi$ da normalização das deltas |
| $K_1(\omega_a,\omega_b)$ | saturação da não linearidade no sítio 1 (seção 3.1) |
| $1/\Gamma_1(\omega_a)\Gamma_1(\omega_b)\Gamma_1(\nu_a)\Gamma_1(\nu_b)$ | um propagador lorentziano para cada fóton que entra ($\nu_{a,b}$) e sai ($\omega_{a,b}$) do sítio 1 |
| $\dfrac{\Gamma_2^{*}(\omega_a)\Gamma_2^{*}(\nu_b)}{\Gamma_2(\omega_a)\Gamma_2(\nu_b)}=t_2(\omega_a)\,t_2(\nu_b)$ | fase linear que os dois fótons que **atravessam** o sítio 2 (como espectador) adquirem |
| $e^{i(\omega_a+\nu_b)L}$ | fase de propagação de caminho entre os sítios para esses dois fótons |

### 3.3 Termo não linear no sítio 2

Análogo, trocando os papéis dos sítios (o sítio 1 vira espectador):

$$
\boxed{\;
T_2=-\,e^{i(\nu_a+\omega_b)L}\;\frac{i\chi_2\gamma_2^{2}}{\pi}\;
\frac{\Gamma_1^{*}(\nu_a)\,\Gamma_1^{*}(\omega_b)}{\Gamma_1(\nu_a)\,\Gamma_1(\omega_b)}\;
\frac{K_2(\omega_a,\omega_b)}{\Gamma_2(\omega_a)\Gamma_2(\omega_b)\Gamma_2(\nu_a)\Gamma_2(\nu_b)}\;}
$$

Note que agora os fótons espectadores do sítio 1 são $\nu_a$ e $\omega_b$ (e não $\omega_a$ e $\nu_b$),
porque, como os fótons são contrapropagantes, cada um encontra os sítios em **ordem inversa**.
A diferença de fase entre $T_1$ e $T_2$, $e^{2i(\nu_a-\omega_a)L}$, é a origem da dependência em $L$.

In [ ]:
def K1(oa, ob, p=P):
    return 1.0 / (1.0 + 2j * p.chi1 / (Gamma1(oa, p) + Gamma1(ob, p)))


def K2(oa, ob, p=P):
    return 1.0 / (1.0 + 2j * p.chi2 / (Gamma2(oa, p) + Gamma2(ob, p)))


def term2_bracket(oa, ob, na, nb, p=P):
    '''T1: não linearidade no sítio 1 (sem a delta de conservação de energia).'''
    phase_L = np.exp(1j * (oa + nb) * p.L)
    return -phase_L * (1j * p.chi1 * p.gamma1 ** 2 / np.pi) \
        * (np.conj(Gamma2(oa, p)) * np.conj(Gamma2(nb, p)) / (Gamma2(oa, p) * Gamma2(nb, p))) \
        * K1(oa, ob, p) \
        / (Gamma1(oa, p) * Gamma1(ob, p) * Gamma1(na, p) * Gamma1(nb, p))


def term3_bracket(oa, ob, na, nb, p=P):
    '''T2: não linearidade no sítio 2 (sem a delta de conservação de energia).'''
    phase_L = np.exp(1j * (na + ob) * p.L)
    return -phase_L * (1j * p.chi2 * p.gamma2 ** 2 / np.pi) \
        * (np.conj(Gamma1(na, p)) * np.conj(Gamma1(ob, p)) / (Gamma1(na, p) * Gamma1(ob, p))) \
        * K2(oa, ob, p) \
        / (Gamma2(oa, p) * Gamma2(ob, p) * Gamma2(na, p) * Gamma2(nb, p))

---
## 4. Amplitude $F=1+G$ e a integral $G$

A quantidade que entra na fidelidade é a **amplitude de sobreposição** do estado de saída, depois de
removida a fase linear, com o estado de entrada:

$$
F=\langle\xi\xi|\,S_1^\dagger(\omega_a)\,S_1^\dagger(\omega_b)\,S\,|\xi\xi\rangle .
$$

Separando $S$ em parte linear e não linear:

* **parte linear**: como $c(\omega)\,s(\omega)=1$ e $\xi$ é normalizada, ela contribui exatamente
  $\int|\xi(\omega_a)|^2d\omega_a\int|\xi(\omega_b)|^2d\omega_b = 1$;
* **parte não linear**: contribui com a integral $G$.

$$
\boxed{\;F=1+G\;}
$$

$$
\boxed{\;
G=\int d\nu_a\,d\nu_b\,d\omega_a\,d\omega_b\;
\xi(\nu_a)\,\xi(\nu_b)\,\xi(\omega_a)\,\xi(\omega_b)\;
c(\omega_a)\,c(\omega_b)\;
\big[T_1+T_2\big]\;\delta(\omega_a+\omega_b-\nu_a-\nu_b)\;}
$$

(As gaussianas $\xi$ são reais, então $\xi^*=\xi$ nas duas amplitudes de saída.)

Se o gate fosse perfeito, $F$ valeria $-1$ em $\phi=\pi$ (fase condicional $\pi$ sem perdas).
$|F|<1$ indica **vazamento** para fora do subespaço computacional e/ou fase diferente de $\pi$.

### 4.1 Eliminando a delta e fatorizando o integrando

A integral tripla efetiva ($\omega_b$ fica fixado pela delta) é cara. Mas o integrando tem uma
estrutura especial. Defina a **energia total**

$$
E=\nu_a+\nu_b=\omega_a+\omega_b .
$$

Como $K_j$ depende só de $E$ (seção 3.1), e todos os outros fatores dependem de **uma** única
frequência cada, o integrando do termo $T_1$ fica

$$
\underbrace{\xi(\omega_a)c(\omega_a)\,\frac{e^{i\omega_a L}t_2(\omega_a)}{\Gamma_1(\omega_a)}}_{f_1(\omega_a)}\;
\underbrace{\xi(\omega_b)c(\omega_b)\,\frac{1}{\Gamma_1(\omega_b)}}_{f_2(\omega_b)}\;
\underbrace{\frac{\xi(\nu_a)}{\Gamma_1(\nu_a)}}_{g_1(\nu_a)}\;
\underbrace{\frac{\xi(\nu_b)\,e^{i\nu_b L}t_2(\nu_b)}{\Gamma_1(\nu_b)}}_{g_2(\nu_b)}\;K_1(E)
$$

e a integral sobre as quatro frequências vinculadas por $E$ vira um produto de **duas convoluções**
seguido de **uma integral unidimensional em $E$**:

$$
\boxed{\;
G_{(1)}=-\frac{i\chi_1\gamma_1^{2}}{\pi}\int dE\;K_1(E)\,
\underbrace{(f_1 * f_2)(E)}_{\text{fótons que saem}}\;
\underbrace{(g_1 * g_2)(E)}_{\text{fótons que entram}}\;}
$$

com $(f*g)(E)=\int d\omega\, f(\omega)\,g(E-\omega)$. Analogamente para o sítio 2:

$$
G_{(2)}=-\frac{i\chi_2\gamma_2^{2}}{\pi}\int dE\;K_2(E)\,
\big(\tilde f_1 * \tilde f_2\big)(E)\,\big(\tilde g_1 * \tilde g_2\big)(E),
$$

$$
\tilde f_1=\frac{\xi\,c}{\Gamma_2},\quad
\tilde f_2=\frac{\xi\,c\,e^{i\omega L}t_1}{\Gamma_2},\quad
\tilde g_1=\frac{\xi\,e^{i\nu L}t_1}{\Gamma_2},\quad
\tilde g_2=\frac{\xi}{\Gamma_2}.
$$

$$
\boxed{\;G=G_{(1)}+G_{(2)}\;}
$$

No código, as convoluções são feitas por FFT (`fftconvolve(a, b) * h`, com $h$ o passo da grade, que
converte a soma discreta em integral) e a integral em $E$ por Simpson (`simpson(..., x=E)`).
O eixo $E$ vai de $2\omega_{\min}$ a $2\omega_{\max}$, pois é a soma de duas frequências da grade.

In [ ]:
def grade(sigma, p=P):
    '''Grade de frequências e seu espaçamento (ver seção 6).'''
    half = p.k_sigma * sigma
    n = int(2 * half / (min(sigma, p.gamma1, p.gamma2) / p.pts_per_width)) + 1
    n = min(max(n, p.n_min), p.n_max) | 1     # ímpar: Simpson exige nº par de intervalos
    w = np.linspace(p.omega_c - half, p.omega_c + half, n)
    return w, w[1] - w[0]


def compute_G(sigma, p=P):
    '''G = G_(1) + G_(2) (seção 4.1): produto de duas convoluções integrado em E.'''
    w, h = grade(sigma, p)

    xi, c = xi_in(w, sigma, p), correction(w, p)
    eL = np.exp(1j * w * p.L)
    G1, G2 = Gamma1(w, p), Gamma2(w, p)
    t1, t2 = np.conj(G1) / G1, np.conj(G2) / G2         # fases de fóton único

    E = np.linspace(2 * w[0], 2 * w[-1], 2 * len(w) - 1)  # energia total
    k1 = 1.0 / (1.0 + 2j * p.chi1 / (p.gamma1 + 1j * (2 * p.Delta1 - E)))
    k2 = 1.0 / (1.0 + 2j * p.chi2 / (p.gamma2 + 1j * (2 * p.Delta2 - E)))

    def conv(a, b):
        return fftconvolve(a, b) * h

    # não linearidade no sítio 1
    G_site1 = -(1j * p.chi1 * p.gamma1 ** 2 / np.pi) * simpson(
        k1 * conv(xi * c * eL * t2 / G1, xi * c / G1)
           * conv(xi / G1, xi * eL * t2 / G1), x=E)

    # não linearidade no sítio 2
    G_site2 = -(1j * p.chi2 * p.gamma2 ** 2 / np.pi) * simpson(
        k2 * conv(xi * c / G2, xi * c * eL * t1 / G2)
           * conv(xi * eL * t1 / G2, xi / G2), x=E)

    return G_site1 + G_site2

---
## 5. Fidelidade média de porta $F_1(\phi)$

A porta implementada age no subespaço computacional de dois qubits ($d=4$) como
$A=\mathrm{diag}(1,1,1,F)$: os três estados $|00\rangle,|01\rangle,|10\rangle$ passam inalterados
(após a correção linear) e apenas $|11\rangle$ — onde os dois fótons interagem — recebe a amplitude $F$.
A porta **alvo** é $U=\mathrm{diag}(1,1,1,e^{-i\phi})$; $\phi=\pi$ dá a CZ.

A fidelidade média de porta, tratando corretamente a perda de probabilidade fora do subespaço, é

$$
F_1=\frac{\big|\mathrm{Tr}(U^\dagger A)\big|^{2}+\mathrm{Tr}(A^\dagger A)}{d\,(d+1)} ,
$$

com $d=4$, $\mathrm{Tr}(U^\dagger A)=3+e^{i\phi}F$ e $\mathrm{Tr}(A^\dagger A)=3+|F|^2$:

$$
|3+e^{i\phi}F|^2 = 9+6\,\mathrm{Re}\!\left(e^{i\phi}F\right)+|F|^2
\quad\Longrightarrow\quad
\boxed{\;F_1(\phi)=\frac{6+3\,\mathrm{Re}\!\left(e^{i\phi}F\right)+|F|^{2}}{10}\;}
$$

**Este é o $F_1(\phi=\pi)$ mostrado no gráfico** (campo `F1_pi` dos arquivos `.npz`):

$$
F_1(\pi)=\frac{6-3\,\mathrm{Re}(F)+|F|^{2}}{10}.
$$

Casos limite úteis para ler o gráfico:

| Situação | $F$ | $F_1(\pi)$ |
|---|---|---|
| sem interação efetiva ($\sigma\to0$, $G\to0$) | $1$ | $(6-3+1)/10=\mathbf{0.4}$ |
| porta perfeita | $-1$ | $(6+3+1)/10=\mathbf{1}$ |

Existe ainda a fidelidade com a **melhor fase** $\phi_{\rm opt}$ (campo `F1_opt`), obtida escolhendo
$\phi$ tal que $e^{i\phi}F=|F|$:

$$
F_1(\phi_{\rm opt})=\frac{6+3|F|+|F|^{2}}{10}.
$$

Como $\mathrm{Re}(e^{i\phi}F)\le|F|$, sempre $F_1(\phi)\le F_1(\phi_{\rm opt})$.

In [ ]:
def F1_fidelity(sigma, p=P):
    G = compute_G(sigma, p)
    F = 1.0 + G
    F1_phi = (6 + 3 * np.real(np.exp(1j * p.phi) * F) + np.abs(F) ** 2) / 10.0   # F1(phi)
    F1_opt = (6 + 3 * np.abs(F) + np.abs(F) ** 2) / 10.0                          # F1(phi_opt)
    return F1_phi, F1_opt, F


# F = 1 (sem interação) deve dar F1(pi) = 0.4 ; F = -1 (porta ideal) deve dar 1
F1_de_F = lambda F, phi=np.pi: (6 + 3 * np.real(np.exp(1j * phi) * F) + abs(F) ** 2) / 10
print("F1(pi | F= 1) =", F1_de_F(1.0))
print("F1(pi | F=-1) =", F1_de_F(-1.0))

---
## 6. Grade numérica de integração

As integrais são feitas em uma grade uniforme de frequências $\omega\in[\omega_c-h_w,\;\omega_c+h_w]$.

**Janela.** As quatro gaussianas confinam o integrando a $|\omega-\omega_c|\lesssim k_\sigma\sigma$
(o núcleo decai só como $1/\omega^4$, mas as gaussianas dominam), então a meia-janela depende apenas de $\sigma$:

$$
h_w = k_\sigma\,\sigma,\qquad k_\sigma=9 .
$$

**Resolução.** O espaçamento precisa resolver a **menor escala** presente, seja a largura do
pacote ($\sigma$) ou a das lorentzianas ($\gamma$). Com `pts_per_width` $=14$ pontos por essa escala:

$$
n=\frac{2\,h_w}{\min(\sigma,\gamma_1,\gamma_2)/14}+1,
\qquad n\in[2001,\;600001]\ \text{e forçado a ser ímpar}
$$

(número ímpar de pontos $\Rightarrow$ número par de intervalos, exigido pela regra de Simpson).
Isso garante convergência tanto para $\sigma\ll\gamma$ quanto para $\sigma\gg\gamma$.

**Varredura em $\sigma$.** O eixo $x$ do gráfico é uma grade **logarítmica**:

$$
\sigma_k = 10^{\,\log_{10}\sigma_{\min}+k\,\Delta},\quad k=0,\dots,99,\quad
\sigma_{\min}=10^{-2},\ \sigma_{\max}=10^{2}
$$

(`np.logspace(-2, 2, 100)`), o que combina com o eixo logarítmico do gráfico.

In [ ]:
sigmas_demo = np.logspace(np.log10(0.01), np.log10(100.0), 100)
print("sigmas: de", sigmas_demo[0], "a", sigmas_demo[-1], "-", len(sigmas_demo), "pontos")
for s in (0.01, 0.1, 1.0, 10.0):
    w, h = grade(s)
    print(f"sigma={s:6.2f}  n={len(w):7d}  janela=±{P.k_sigma*s:6.2f}  passo h={h:.2e}")

---
## 7. Verificação: recalcular e comparar com os dados salvos

Os dados usados no gráfico foram gerados por `calcular.py` e estão em `data/counterprop/*.npz`.
Como as funções acima são uma versão parametrizada do mesmo cálculo, elas devem reproduzir os
valores salvos. Recalculamos $F_1(\pi)$ em alguns valores de $\sigma$ (os do próprio arquivo) e comparamos.

> A varredura completa (100 valores de $\sigma$ × 6 curvas, com grades de até 600 001 pontos em $\sigma$ pequeno)
> é lenta, por isso aqui só refazemos alguns pontos.

In [ ]:
DATA_DIR = os.path.join('data', 'counterprop')

# (L/gamma, arquivo, cor) — as 6 curvas de plot.py
CURVAS = [
    (0.0, 'teste_fidelity_counterprop_l0_chi1000_gamma1_omega0_min001.npz',  '#8A2BE2'),
    (0.4, 'teste_fidelity_counterprop_l04_chi1000_gamma1_omega0_min001.npz', '#FF0000'),
    (0.6, 'teste_fidelity_counterprop_l06_chi1000_gamma1_omega0_min001.npz', '#3CB371'),
    (0.8, 'teste_fidelity_counterprop_l08_chi1000_gamma1_omega0_min001.npz', '#FF8C00'),
    (1.0, 'teste_fidelity_counterprop_l1_chi1000_gamma1_omega0_min001.npz',  '#1E90FF'),
    (2.0, 'teste_fidelity_counterprop_l2_chi1000_gamma1_omega0_min001.npz',  'black'),
]

Pv = Params()
print(f"{'L':>4} {'sigma':>8} {'F1(pi) salvo':>14} {'F1(pi) recalc.':>16} {'|dif|':>10}")
for L_val, fname, _ in CURVAS:
    d = np.load(os.path.join(DATA_DIR, fname))
    Pv.L = L_val
    for idx in (55, 70, 85):                 # três valores de sigma na região de interesse
        s = d['sigmas'][idx]
        f1_pi, _, _ = F1_fidelity(s, Pv)
        print(f"{L_val:4.1f} {s:8.3f} {d['F1_pi'][idx]:14.8f} {f1_pi:16.8f} {abs(f1_pi - d['F1_pi'][idx]):10.2e}")

In [ ]:
# Consistência entre o rótulo L/gamma da curva e o valor de L gravado dentro do .npz
for L_val, fname, _ in CURVAS:
    L_arq = float(np.load(os.path.join(DATA_DIR, fname))['L'])
    if not np.isclose(L_arq, L_val):
        print(f"ATENÇÃO: {fname}\n         rotulado L/gamma={L_val}, mas o arquivo foi gerado com L={L_arq}")
print("checagem concluída")

> **Nota sobre o estado atual dos dados.** Se a checagem acima imprimir uma `ATENÇÃO`, o arquivo daquela curva
> foi gerado por `calcular.py` com um `L` diferente do indicado no nome/rótulo (por exemplo, esquecendo de
> atualizar `L` antes de rodar). Nesse caso, a curva aparece no gráfico da seção 8 com o rótulo errado. A
> coluna "|dif|" da tabela acima também denuncia o problema: o valor recalculado com o $L$ do rótulo não
> reproduz o arquivo. Para corrigir, rode `calcular.py` com o `L` correto e sobrescreva o `.npz`.

---
## 8. O gráfico, elemento por elemento

O gráfico é $F_1(\pi)$ (eixo $y$) em função de $\sigma$ (eixo $x$, logarítmico), com uma curva por
valor de $L/\gamma$. Cada elemento visual de `plot.py` está descrito abaixo.

| Elemento | Código em `plot.py` | Expressão / significado |
|---|---|---|
| Eixo $x$ | `xlabel = r'Largura do pacote de onda $\sigma$'` | $\sigma$, largura espectral de $\xi(\omega)$ (seção 2.3); em unidades de $\gamma$ |
| Escala do eixo $x$ | `ax.semilogx(...)` | logarítmica, casa com a grade `np.logspace` da seção 6 |
| Faixa do eixo $x$ | `xlim=(1e-2, 1e1)` | $10^{-2}\le\sigma\le10^{1}$ (os dados vão até $10^2$, mas o gráfico corta em $10^1$) |
| Eixo $y$ | `ylabel = 'Fidelidade $F_1$'` | $F_1(\phi=\pi)=\dfrac{6-3\,\mathrm{Re}F+|F|^2}{10}$ (seção 5) |
| Faixa do eixo $y$ | `ylim=(0.3, 1.02)` | o piso teórico sem interação é $0.4$; o teto ideal é $1$ |
| Curvas | `field='F1_pi'` | campo `F1_pi` dos `.npz` (com `F1_opt` seria $F_1(\phi_{\rm opt})$) |
| Legenda | `label = r'$\frac{L}{\gamma}=0.4$'` | separação entre os sítios em unidades de $1/\gamma$ (como $\gamma=1$, é o próprio $L$) |
| Título | `r'$\chi=1000,\ \gamma=1,\ \omega_0=0$'` | parâmetros fixos comuns a todas as curvas |
| Estilo | `marker='o-'`, `markeredgewidth=2.7`, `marker_alpha=0.5` | linha fina passando por marcadores pequenos com borda larga e semitransparente, o que dá o aspecto de "faixa" contínua |
| Fonte | `mathtext.fontset='cm'`, `font.family='serif'` | Computer Modern, estilo LaTeX |
| SVG | `svg.fonttype='path'` | texto convertido em curvas: o SVG não depende de fontes instaladas |

**Leitura física do resultado.**

* $\sigma\to0$ (pacote muito longo no tempo): $G\propto\sigma/\gamma\to0$, então $F\to1$ e $F_1(\pi)\to0.4$.
  Fótons contrapropagantes num sítio pontual não acumulam fase condicional para pacotes longos.
* $\sigma\sim\gamma$: pico de fidelidade — pacote comparável à largura do sítio.
* $\sigma\gg\gamma$: o pacote é mais curto que o tempo de resposta do sítio; a fase condicional
  acumulada se degrada de novo.
* $L$ altera o **relativo** entre $T_1$ e $T_2$ pelo fator $e^{2i(\nu_a-\omega_a)L}$ (seção 3.3), deslocando
  e modificando o pico.

In [ ]:
# ---- Configuração global de estilo (mesma de plot.py) ----
plt.rcParams.update({
    'figure.figsize': (6, 4.5),
    'font.size': 10,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'axes.linewidth': 1.0,
    'svg.fonttype': 'path',
})


def plot_curve(ax, infile, field, color, label, marker='o-', linewidth=.5,
               markersize=1, markeredgewidth=2.7, marker_alpha=0.5):
    data = np.load(infile)
    rgba = to_rgba(color, marker_alpha)      # cor com transparência para a borda/preenchimento
    ax.semilogx(                              # eixo x logarítmico
        data['sigmas'], data[field], marker,
        color=color, linewidth=linewidth, markersize=markersize,
        markeredgewidth=markeredgewidth, markeredgecolor=rgba, markerfacecolor=rgba,
        label=label,
    )
    return data


fig, ax = plt.subplots(figsize=(7, 5))
for L_val, fname, color in CURVAS:
    plot_curve(ax, os.path.join(DATA_DIR, fname), 'F1_pi', color,
               label=rf'$\frac{{L}}{{\gamma}}={L_val:g}$')

ax.set_xlabel(r'Largura do pacote de onda $\sigma$')       # eixo x
ax.set_ylabel('Fidelidade $F_1$')                          # eixo y
ax.set_title(r'$\chi=1000,\ \gamma=1,\ \omega_0=0$')       # parâmetros fixos
ax.set_ylim(0.3, 1.02)
ax.set_xlim(1e-2, 1e1)
ax.legend(loc='upper right')
fig.tight_layout()
# fig.savefig('fidelity_all.svg', dpi=300, bbox_inches='tight')
plt.show()

---
## 9. Checagem analítica: limite $\sigma\to0$

Para $\sigma\to0$, o pacote é estreito em frequência e o núcleo (que varia na escala de $\gamma$) pode ser
avaliado em $\omega_c$ e retirado da integral; o que sobra é a integral das gaussianas, proporcional a $\sigma$.
O resultado (deduzido analiticamente e confirmado numericamente, veja abaixo) é

$$
G(\sigma\to0)\;\simeq\;2\sqrt{\pi}\,\sigma\left[
-\frac{i\chi_1\gamma_1^{2}}{\pi}\,\frac{\mathcal K_1}{|\Gamma_1(\omega_c)|^{4}}
-\frac{i\chi_2\gamma_2^{2}}{\pi}\,\frac{\mathcal K_2}{|\Gamma_2(\omega_c)|^{4}}\right],
\qquad
\mathcal K_j=\frac{\Gamma_j(\omega_c)}{\Gamma_j(\omega_c)+i\chi_j}.
$$

Aqui $\mathcal K_j=K_j(\omega_c,\omega_c)$ é a saturação avaliada em $E=2\omega_c$
(seção 3.1: $\Gamma_j(\omega_c)+\Gamma_j(\omega_c)=2\Gamma_j(\omega_c)$).

Duas consequências:

1. $G\propto\sigma$: $F\to1$ e $F_1(\pi)\to0.4$ linearmente em $\sigma$;
2. **$G$ não depende de $L$**: para pacotes muito longos, a separação entre os sítios é irrelevante,
   e por isso **todas as curvas colapsam** na mesma assíntota à esquerda do gráfico.

A expansão só vale para $\sigma\ll\gamma$ (aqui $\gamma=1$): os dados se afastam da assíntota conforme $\sigma$
se aproxima de $\sim0.1$, onde termos de ordem superior em $\sigma/\gamma$ deixam de ser desprezíveis.
Abaixo, comparamos a assíntota com os dados de cada curva.

In [ ]:
def G_sigma_zero(sigma, p=P):
    '''Limite analítico de G para sigma -> 0.'''
    wc = p.omega_c
    def termo(chi, gamma, Gam):
        Kc = Gam(wc, p) / (Gam(wc, p) + 1j * chi)
        return -(1j * chi * gamma ** 2 / np.pi) * Kc / np.abs(Gam(wc, p)) ** 4
    return 2 * np.sqrt(np.pi) * sigma * (termo(p.chi1, p.gamma1, Gamma1) + termo(p.chi2, p.gamma2, Gamma2))


def F1_pi_assintota(sigma):
    F = 1.0 + G_sigma_zero(sigma)
    return (6 - 3 * np.real(F) + np.abs(F) ** 2) / 10


s_as = np.logspace(-2, -1, 60)
fig, ax = plt.subplots(figsize=(7, 5))
for L_val, fname, color in CURVAS:
    d = np.load(os.path.join(DATA_DIR, fname))
    m = d['sigmas'] <= 0.1
    ax.semilogx(d['sigmas'][m], d['F1_pi'][m], 'o', color=color, ms=3, alpha=0.6,
                label=rf'$L/\gamma={L_val:g}$ (numérico)')
ax.semilogx(s_as, F1_pi_assintota(s_as), 'k--', lw=1.2, label=r'assíntota $\sigma\to0$')
ax.set_xlabel(r'Largura do pacote de onda $\sigma$')
ax.set_ylabel('Fidelidade $F_1$')
ax.set_title(r'Comparação com o limite analítico $\sigma\to0$')
ax.set_ylim(0.4, 0.8)
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout()
plt.show()

print("F1(pi) em sigma=0.01 (assíntota):", F1_pi_assintota(0.01))
for L_val, fname, _ in CURVAS:
    d = np.load(os.path.join(DATA_DIR, fname))
    print(f"L={L_val:3.1f}  F1(pi)[sigma={d['sigmas'][0]:.3f}] = {d['F1_pi'][0]:.6f}")